In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import os
import torch
import torch.nn as nn
import time
import pandas as pd
from scipy.stats import pearsonr

In [3]:
from model.util import Normalizer
from model.database_util import get_hist_file, get_job_table_sample, collator
from model.model import QueryFormer
from model.database_util import Encoding
from model.dataset import PlanTreeDataset
from model.trainer import eval_workload, train

In [4]:
data_path = './data/imdb/'

In [5]:
class Args:
    # bs = 1024
    # SQ: smaller batch size
    bs = 128
    lr = 0.001
    # epochs = 200
    epochs = 100
    clip_size = 50
    embed_size = 64
    pred_hid = 128
    ffn_dim = 128
    head_size = 12
    n_layers = 8
    dropout = 0.1
    sch_decay = 0.6
    device = 'cuda:0'
    newpath = './results/full/cost/'
    to_predict = 'cost'
args = Args()

import os
if not os.path.exists(args.newpath):
    os.makedirs(args.newpath)

In [6]:
hist_file = get_hist_file(data_path + 'histogram_string.csv')
cost_norm = Normalizer(-3.61192, 12.290855)
card_norm = Normalizer(1,100)

/home/AiChaosN/Project/Phd/project/QueryFormer_VLDB2022/model/database_util.py:76: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  hist_file['freq'][i] = freq_np
/home/AiChaosN/Project/Phd/project/QueryFormer_VLDB2022/model/database_util.py:89

In [7]:
encoding_ckpt = torch.load('checkpoints/encoding.pt')
encoding = encoding_ckpt['encoding']
checkpoint = torch.load('checkpoints/cost_model.pt', map_location='cpu')

In [8]:
from model.util import seed_everything
seed_everything()

In [9]:
model = QueryFormer(emb_size = args.embed_size ,ffn_dim = args.ffn_dim, head_size = args.head_size, \
                 dropout = args.dropout, n_layers = args.n_layers, \
                 use_sample = True, use_hist = True, \
                 pred_hid = args.pred_hid
                )

In [10]:
_ = model.to(args.device)

In [11]:
to_predict = 'cost'

In [12]:
## 训练数据集和验证数据集
imdb_path = './data/imdb/'
dfs = []  # list to hold DataFrames
# SQ: added
# for i in range(2):
for i in range(18):
    file = imdb_path + 'plan_and_cost/train_plan_part{}.csv'.format(i)
    df = pd.read_csv(file)
    dfs.append(df)

full_train_df = pd.concat(dfs)

val_dfs = []  # list to hold DataFrames
for i in range(18,20):
    file = imdb_path + 'plan_and_cost/train_plan_part{}.csv'.format(i)
    df = pd.read_csv(file)
    val_dfs.append(df)

val_df = pd.concat(val_dfs)

In [13]:
table_sample = get_job_table_sample(imdb_path+'train')

Loaded queries with len  100000
Loaded bitmaps


In [14]:
train_ds = PlanTreeDataset(full_train_df, None, encoding, hist_file, card_norm, cost_norm, to_predict, table_sample)
val_ds = PlanTreeDataset(val_df, None, encoding, hist_file, card_norm, cost_norm, to_predict, table_sample)

In [15]:
train_ds.json_df
train_ds.table_sample
train_ds.encoding
train_ds.hist_file

print(len(train_ds))
print(dir(train_ds))
x, y = train_ds[0]
print(x.keys())
print(x["x"].shape)
print(y)

print("解析df")
print(train_ds.json_df.head())

# 每个plan的cost
print("每个plan的cost")
print(len(train_ds.costs))
print(train_ds.costs[0])

# 每个plan的card
print("每个plan的card")
print(len(train_ds.cards))
print(train_ds.cards[0])

# 每个plan的encoding
print("每个plan的encoding")
print(train_ds.encoding)



90000
['__add__', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'calculate_height', 'card_labels', 'cards', 'collated_dicts', 'cost_labels', 'costs', 'demo_complete_pipeline', 'demo_feature_encoding', 'demo_json_to_tree_conversion', 'demo_tree_structure_visualization', 'demo_with_sample_data', 'encoding', 'gts', 'hist_file', 'js_node2dict', 'json_df', 'labels', 'length', 'node2dict', 'old_getitem', 'pre_collate', 'table_sample', 'to_predict', 'topo_sort', 'traversePlan', 'treeNodes']
dict_keys(['x', 'attn_bias', 'rel_pos', 'heights'])
torch.Size([1, 30, 1165])
(tensor(0.6348, d

In [ ]:
crit = nn.MSELoss()
model, best_path = train(model, train_ds, val_ds, crit, cost_norm, args)

# 结果 1.12~1.2左右
# 时间周期长(参数量大)

Epoch: 0  Avg Loss: 3.54807106745688e-05, Time: 69.91508674621582
Median: 1.4462946411026683
Mean: 45.74551720621151
Training Q-error - Median: 1.4463, 90th: 4.8910, Mean: 45.7455
Epoch: 20  Avg Loss: 1.2642700157852637e-05, Time: 1280.4934649467468
Median: 1.3186476931272622
Mean: 1.8516990645912799
Training Q-error - Median: 1.3186, 90th: 2.7577, Mean: 1.8517
Epoch: 40  Avg Loss: 1.0573475080309436e-05, Time: 2536.3108298778534
Median: 1.239983145692563
Mean: 1.7245060505957281
Training Q-error - Median: 1.2400, 90th: 2.4508, Mean: 1.7245
Median: 1.1910974726217742
Mean: 1.7201419842355499
Validation Q-error - Median: 1.1911, Mean: 1.7201
Median: 1.2072344945280522
Mean: 1.7176158661259486
Validation Q-error - Median: 1.2072, Mean: 1.7176
Median: 1.1917345980658371
Mean: 1.7282983976160637
Validation Q-error - Median: 1.1917, Mean: 1.7283
Median: 1.2248461417983862
Mean: 1.735289370112439
Validation Q-error - Median: 1.2248, Mean: 1.7353
Median: 1.2398347655728212
Mean: 1.74365207798

In [17]:
methods = {
    'get_sample' : get_job_table_sample,
    'encoding': encoding,
    'cost_norm': cost_norm,
    'hist_file': hist_file,
    'model': model,
    'device': args.device,
    'bs': 512,
}

In [18]:
_ = eval_workload('job-light', methods)

Loaded queries with len  70
Loaded bitmaps
Median: 1.645763712452562
Mean: 15.05386870860079
Corr:  0.891513577975937
Median: 1.645763712452562
Mean: 15.05386870860079
Q-error metrics for job-light:
  Q-median: 1.6458
  Q-90th: 27.3540
  Q-mean: 15.0539


In [19]:
_ = eval_workload('synthetic', methods)

Loaded queries with len  5000
Loaded bitmaps
Median: 1.1187303290925152
Mean: 1.5646509101927666
Corr:  0.9859175035438174
Median: 1.1187303290925152
Mean: 1.5646509101927666
Q-error metrics for synthetic:
  Q-median: 1.1187
  Q-90th: 2.0252
  Q-mean: 1.5647
